# Pretrain a small language model with TrainLM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/Dhiraj309/TrainLM/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb)

Open a copy in Colab or import it into Kaggle from the button above. Choose a runtime with the hardware you need; the link does not provision a TPU.

The setup cell does not install or replace PyTorch/TorchTPU. Before a native TPU run, it reports installed `torch-tpu` and `libtpu` versions. TrainLM checks package compatibility before loading data. Each run prints JSON provenance with the exact source commit, runtime package versions, and accelerator discovery fields; keep it with the report.

Open this notebook on a **single-device native TorchTPU runtime**. The fresh-run cell builds a randomly initialized Hugging Face GPT-2, learns a tokenizer from training text, trains, evaluates, and exports the model. The default 10,000-token run is only a workflow smoke test. It does not produce a useful model or certify TPU performance. For a longer run, increase `FRESH_MAX_TOKENS`; if preparation reports too few packed sequences, also increase `FRESH_MAX_TRAIN_EXAMPLES`. Resume automatically adds 2,000 tokens to the saved target. The artifact cell reloads the export with Transformers on CPU and prints a short generation preview; smoke output is not a quality result.

**Model architecture:** `model_type="gpt2"` selects Hugging Face GPT-2. TrainLM creates `GPT2LMHeadModel` with random weights; it does not load a pretrained GPT-2 checkpoint.

Run setup, then the fresh-run cell. To continue in the same session, change `RUN_RESUME = False` to `True` in the resume cell and run only that cell. To check recovery after a Python kernel/process restart, keep the same VM and run directory, then run only the resume cell; it defines its own `RUN_DIR` and imports the API. Do not rerun setup or fresh training. If the VM was replaced, restore the full run directory and compatible packages first. The cell validates the saved report and resumes with a 12,000-token budget. After resume, rerun the artifact cell to preview and bundle the updated report and export.

For a new independent run, use an unused `RUN_DIR`. Advanced settings and checkpoint recovery: [pretraining guide](../README.md#start-here-pretrain-a-language-model-from-scratch).


In [ ]:
import subprocess
import sys

source = "trainlm @ git+https://github.com/Dhiraj309/TrainLM.git@main"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", source],
    check=True,
)
requirements = [
    "huggingface-hub>=1.5,<2",
    "PyYAML>=6,<7",
    "transformers>=5.15,<6",
    "datasets>=3,<5",
    "tokenizers>=0.21,<1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("TrainLM installed without asking pip to replace PyTorch or TorchTPU.")
from importlib.metadata import PackageNotFoundError, version

for package_name in ("torch-tpu", "libtpu"):
    try:
        print(f"{package_name}=={version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}=MISSING; native TPU needs a runtime that provides TorchTPU")


In [ ]:
import json
from importlib.metadata import distribution
from trainlm import TrainLMPretrainingConfig, run_pretraining
direct_url = json.loads(distribution("trainlm").read_text("direct_url.json") or "{}")
source_revision = direct_url.get("vcs_info", {}).get("commit_id")
assert source_revision, "Reinstall TrainLM from Git to record its exact commit."
RUN_DIR = "trainlm-pretraining-fresh"
FRESH_MAX_TOKENS = 10_000  # Fresh run: max_tokens=10_000.
FRESH_MAX_TRAIN_EXAMPLES = 2_000  # Raise if scaling the token budget.
config = TrainLMPretrainingConfig(
    output_dir=RUN_DIR,
    max_tokens=FRESH_MAX_TOKENS,
    max_train_examples=FRESH_MAX_TRAIN_EXAMPLES,
    sequence_length=128,
    model_type="gpt2",  # Randomly initialized Hugging Face GPT-2.
    model_embedding_size=256,
    model_layers=4,
    model_attention_heads=4,
    accelerator="native_tpu",
)
result = run_pretraining(config, show_progress=True, source_revision=source_revision)
provenance = {key: result.report[key] for key in ("source_revision", "runtime", "accelerator")}
print(json.dumps(provenance, sort_keys=True))
print(result.summary())
print(f"checkpoint: {result.checkpoint_path}")
print(f"report: {result.report_path}")
print(f"model: {result.export_path}")


In [ ]:
import json
from importlib.metadata import distribution
from dataclasses import replace
from trainlm import restore_pretraining_report, run_pretraining
direct_url = json.loads(distribution("trainlm").read_text("direct_url.json") or "{}")
source_revision = direct_url.get("vcs_info", {}).get("commit_id")
assert source_revision, "Reinstall TrainLM from Git to record its exact commit."
RUN_DIR = "trainlm-pretraining-fresh"
RUN_RESUME = False  # Set True, then run only this cell after a kernel restart.
if RUN_RESUME:
    saved = restore_pretraining_report(f"{RUN_DIR}/report.json")
    RESUME_MAX_TOKENS = saved.config.max_tokens + 2_000
    resume_config = replace(saved.config, max_tokens=RESUME_MAX_TOKENS)
    resumed = run_pretraining(
        resume_config,
        resume_from_checkpoint=saved.checkpoint_path,
        extend_training_budget=True,
        show_progress=True,
        source_revision=source_revision,
        report_path=f"{RUN_DIR}/resume-report.json",
        export_path=f"{RUN_DIR}/resumed-export",
    )
    provenance = {key: resumed.report[key] for key in ("source_revision", "runtime", "accelerator")}
    print(json.dumps(provenance, sort_keys=True))
    print(resumed.summary())
    print(f"resumed_from: {saved.checkpoint_path}")
    print(f"report: {resumed.report_path}")
    print(f"model: {resumed.export_path}")
else:
    print("Fresh run is the default. Set RUN_RESUME = True to continue.")


In [ ]:
from trainlm import bundle_pretraining_artifacts
RUN_RESUME = globals().get("RUN_RESUME", False)
run = resumed if RUN_RESUME else result
model_path = getattr(run, "export_path", None)
if model_path:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
    model.eval()
    inputs = tokenizer("Once upon a time", return_tensors="pt")
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=24, do_sample=False)
    print(tokenizer.decode(output[0], skip_special_tokens=True))

if RUN_RESUME:
    bundle = bundle_pretraining_artifacts(resumed.report_path)
else:
    bundle = bundle_pretraining_artifacts(result.report_path)
try:
    from google.colab import files
except ImportError:
    print(f"Artifact bundle: {bundle}")
else:
    files.download(str(bundle))
